In [1]:
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout, BatchNormalization
from tensorflow.keras.preprocessing import image
from tensorflow import keras
import numpy as np
import tkinter as tk
from tkinter import filedialog
from PIL import Image, ImageTk

In [2]:
# # Loading ImageNet 32x32, resized version of ImageNet without liscencing yap
# dataset, info = tfds.load('imagenet_resized/32x32', with_info=True, as_supervised=True)

# print(info)

(ds_train, ds_test), ds_info = tfds.load(
    'mnist',
    split=['train', 'test'],
    shuffle_files=True,
    as_supervised=True,
    with_info=True,
)

print(ds_info)

# train_dataset = dataset['train'] # Train data set
# validation_dataset = dataset['validation']  # Validation data set

train_dataset = ds_train
validation_dataset = ds_test

tfds.core.DatasetInfo(
    name='mnist',
    full_name='mnist/3.0.1',
    description="""
    The MNIST database of handwritten digits.
    """,
    homepage='http://yann.lecun.com/exdb/mnist/',
    data_path='/Users/cai_ya/tensorflow_datasets/mnist/3.0.1',
    file_format=tfrecord,
    download_size=11.06 MiB,
    dataset_size=21.00 MiB,
    features=FeaturesDict({
        'image': Image(shape=(28, 28, 1), dtype=uint8),
        'label': ClassLabel(shape=(), dtype=int64, num_classes=10),
    }),
    supervised_keys=('image', 'label'),
    disable_shuffling=False,
    splits={
        'test': <SplitInfo num_examples=10000, num_shards=1>,
        'train': <SplitInfo num_examples=60000, num_shards=1>,
    },
    citation="""@article{lecun2010mnist,
      title={MNIST handwritten digit database},
      author={LeCun, Yann and Cortes, Corinna and Burges, CJ},
      journal={ATT Labs [Online]. Available: http://yann.lecun.com/exdb/mnist},
      volume={2},
      year={2010}
    }""",
)


In [3]:
# Check if the training generator is yielding data
for data_batch, labels_batch in train_dataset:
    print('Data batch shape:', data_batch.shape)
    print('Labels batch shape:', labels_batch.shape)
    break  # Stop after one batch to avoid printing too much

Data batch shape: (28, 28, 1)
Labels batch shape: ()


2024-09-19 12:44:19.084686: W tensorflow/core/kernels/data/cache_dataset_ops.cc:913] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


In [4]:
train_dataset = ds_train
validation_dataset = ds_test

# OPTIONAL: View shape of a batch
for image, label in train_dataset.take(10):
    print(image.numpy().shape, label.numpy())

#Input shape = shape of image
# numclass = output layer size

def model(input_shape=(28, 28, 1), num_classes=10):
  model = keras.Sequential()

  model.add(Conv2D(filters=32, kernel_size=(3, 3), strides=(1, 1), activation='relu', padding='same'))
  model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))

  model.add(Conv2D(filters=64, kernel_size=(3, 3), strides=(1, 1), activation='relu', padding='same'))
  model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))

  model.add(Flatten())
  model.add(Dropout(0.5))

  model.add(Dense(num_classes, activation='softmax'))

  return model

model = model()

(28, 28, 1) 4
(28, 28, 1) 1
(28, 28, 1) 0
(28, 28, 1) 7
(28, 28, 1) 8
(28, 28, 1) 1
(28, 28, 1) 2
(28, 28, 1) 7
(28, 28, 1) 1
(28, 28, 1) 6


2024-09-19 12:44:21.285747: W tensorflow/core/kernels/data/cache_dataset_ops.cc:913] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.
2024-09-19 12:44:21.286401: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [5]:
# Compile the model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [6]:
# Train the model

history = model.fit(
    train_dataset.batch(8)
        .map(lambda x, y: (tf.cast(x, tf.float64) / 255.0, tf.one_hot(tf.cast(y, tf.int32), depth=10)))  # Cast images to float32 and scale
        .repeat(),
    steps_per_epoch=len(train_dataset),
    validation_data=validation_dataset.batch(8)
        .map(lambda x, y: (tf.cast(x, tf.float64) / 255.0, tf.one_hot(tf.cast(y, tf.int32), depth=10)))  # Cast images to float32 and scale
        .repeat(),
    validation_steps=len(validation_dataset),
    epochs=10
)

Epoch 1/10
60000/60000 ━━━━━━━━━━━━━━━━━━━━ 331s 6ms/step - accuracy: 0.9654 - loss: 0.1088 - val_accuracy: 0.9900 - val_loss: 0.0313
Epoch 2/10
60000/60000 ━━━━━━━━━━━━━━━━━━━━ 280s 5ms/step - accuracy: 0.9902 - loss: 0.0305 - val_accuracy: 0.9917 - val_loss: 0.0308
Epoch 3/10
60000/60000 ━━━━━━━━━━━━━━━━━━━━ 279s 5ms/step - accuracy: 0.9923 - loss: 0.0241 - val_accuracy: 0.9906 - val_loss: 0.0364
Epoch 4/10
37666/60000 ━━━━━━━━━━━━━━━━━━━━ 1:35 4ms/step - accuracy: 0.9930 - loss: 0.0231

KeyboardInterrupt: 

In [2]:
model.save('M1-Float64.h5')

NameError: name 'model' is not defined